In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys

sys.path.append("..")
from src.utils import plot_functions as pf
from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd

In [ ]:
sns.set_style("whitegrid")

In [ ]:
ds = pf.open_experiments(
    experiments_csv="../evaluation/final_runs_swh_1h.csv",
    boundary_nodes_csv="../src/data/node_selection/boundary_nodes.csv",
)

In [ ]:
metrics = pf.compute_binned_scores(ds, variable="sigWaveHeight", bins=[0, 0.5, 1.0, 2.0, 4.0, 100])

In [ ]:
dmetrics = pf.compute_binned_change_scores(ds, variable="sigWaveHeight", bins=[-2, -1, -0.5, 0, 0.5, 1, 2])
dmetrics.head()

In [ ]:
fig = plt.figure(figsize=(12, 4))
pf.plot_overall_scores(metrics, metric="rmse", fig=fig);

In [ ]:
fig = plt.figure(figsize=(12, 4))
pf.plot_overall_scores(metrics.query("experiment != 'densenet'"), metric="rmse", fig=fig);

In [ ]:
metrics

In [ ]:
metrics.query("bin == 'overall'").sort_values("rmse", ascending=True)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 4))
sns.barplot(
    data=metrics.query("bin != 'overall'").query("experiment != 'densenet'"), x="bin", y="rmse", hue="experiment"
)
ax.legend(ncol=4, loc=2)
ax.set_xlabel("Significant wave height (m)")
ax.set_xticks([0, 1, 2, 3, 4])
ax.set_xticklabels(["0 - 0.5", "0.5 - 1.0", "1.0 - 2.0", "2.0 - 4.0", "> 4.0"])
ax.set_ylabel("RMSE significant wave height (m)")
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 4))
sns.barplot(
    data=dmetrics.query("bin != 'overall'").query("experiment != 'densenet'"), x="bin", y="rmse", hue="experiment"
)
ax.legend(ncol=4, loc=2)
ax.set_xlabel("Change in significant wave height (m)")
# ax.set_xticks([0, 1, 2, 3, 4])
# ax.set_xticklabels(["0 - 0.5", "0.5 - 1.0", "1.0 - 2.0", "2.0 - 4.0", "> 4.0"])
ax.set_ylabel("RMSE significant wave height (m)")
fig.tight_layout()

In [ ]:
fig = pf.plot_architecture_scores(metrics, architecture="graph")
fig.gca().set_xlabel("Number of nodes");

In [ ]:
fig = pf.plot_architecture_scores(metrics, architecture="graph-lstm");
# fig.gca().set_xlabel("Number of nodes");

In [ ]:
fig = pf.plot_architecture_scores(metrics, architecture="lstm", metric="rmse");

In [ ]:
ds.isel(time=slice(95, 95 + 24)).time.values[[0, -1]]

In [ ]:
ds.isel(time=slice(95 + 13 * 24, 95 + 14 * 24))["sigWaveHeight"].mean().values

In [ ]:
ds.isel(experiment=-1, time=slice(None, None, 24))["sigWaveHeight"].mean(dim="nSCHISM_hgrid_node").plot()

In [ ]:
pf.plot_spatial_error(
    ds,
    experiment=["lstm-6", "graph-lstm-6-50", "graph-lstm-6-500"] + ["lstm-48", "graph-lstm-48-50", "graph-lstm-48-500"],
    variable="sigWaveHeight",
    time_indices=(119 + 10 * 24, 119 + 10 * 24 + 24),
);

In [ ]:
## Test set analysis

In [ ]:
smax = ds.sel(experiment="ground_truth")["sigWaveHeight"].max(dim="nSCHISM_hgrid_node")
smax.plot()

In [ ]:
ds.sel(experiment="ground_truth")["sigWaveHeight"].sel(time=slice("2020-12-06", "2020-12-06"))

In [ ]:
benchmark_cpu = pd.read_csv("./benchmark_inference_timing_cpu_singlethread.csv")
benchmark_cpu

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 4))
sns.barplot(
    data=benchmark_cpu,
    x="label",
    y="cpu_mean_ms",
)
xt = ax.get_xticklabels()
ax.set_xticks(range(len(xt)))
ax.set_xticklabels(metrics.experiment.unique(), rotation=45)
ax.set_xlabel("")
ax.set_ylabel("Mean execution time on CPU (single thread) (ms)")
ax.set_title("For 500 nodes (LSTM: 500 samples / Graph with 50 nodes: 10 samples etc.")
fig.tight_layout()

In [ ]:
ads = pf.open_experiments(
    experiments_csv="../evaluation/final_runs_all_1h.csv",
    boundary_nodes_csv="../src/data/node_selection/boundary_nodes.csv",
)

In [ ]:
ads

In [ ]:
smetrics = pf.compute_binned_scores(ads, variable="sigWaveHeight", bins=[0, 0.5, 1.0, 2.0, 4.0, 100])

In [ ]:
smetrics["version"] = "all"

In [ ]:
metrics["version"] = "swh"

In [ ]:
cmetrics = pd.concat([metrics, smetrics])  # .query("bin = 'overall'")
cmetrics.query("bin == 'overall'")

In [ ]:
common = [
    "lstm-6",
    "lstm-48",
    "graph-50",
    "graph-500",
    "graph-lstm-6-50",
    "graph-lstm-48-50",
    "graph-lstm-6-500",
    "graph-lstm-48-500",
]

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 4))
sns.barplot(
    data=cmetrics.query("bin == 'overall'").query("experiment in @common"), x="experiment", y="rmse", hue="version"
)
ax.set_ylabel("RMSE significant wave height (m)")
fig.tight_layout()

In [ ]:
pmetrics = pf.compute_binned_scores(ads, variable="meanWavePeriod", bins=[0, 1, 3, 7])
pmetrics

In [ ]:
fig = pf.plot_overall_scores(pmetrics.query("experiment != 'densenet-0'"))
ax = fig.gca()
ax.set_ylabel("RMSE mean wave period");

In [ ]:
dmetrics = pf.compute_binned_scores(ads, variable="meanWaveDirection", bins=[0, 90, 180, 270, 360], circular=True)
dmetrics.head()

In [ ]:
fig = pf.plot_overall_scores(dmetrics.query("experiment != 'densenet-0'"))
ax = fig.gca()
ax.set_ylabel("RMSE mean wave direction");

In [ ]:
pf.plot_spatial_error(
    ads,
    experiment=["lstm-6", "graph-lstm-6-50", "graph-lstm-6-500"] + ["lstm-48", "graph-lstm-48-50", "graph-lstm-48-500"],
    variable="meanWavePeriod",
    time_indices=(119 + 10 * 24, 119 + 10 * 24 + 24),
);